RoPE implementation from scratch

Build Rotary Position Embeddings to understand how modern LLMs encode position




In [6]:
# rope (rotatory position embeddings from scratch)
# used by llama, mistral, quen

import torch
import torch.nn as nn

def precompute_rope_frequencies(dim, max_seq_len, base=10000):
    # pre compute rotation frequencies for rope
    # each pair of dimensions gets a different frequency

    # frequencies decrease as dimesnion index increases
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))

    # position from 0 to max_seq_len
    positions = torch.arange(max_seq_len).float()

    # outer product (max_seq_len, dim/2)
    angles = positions.unsqueeze(1) * freqs.unsqueeze(0)

    # return cos and sin for rotation
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    # apply rotatory embeddings to input tensor, 
    # x: [batch, seq_len, num_heads, head_dim)]

    # split into pairs for rotation
    x1, x2 = x[..., ::2], x[...,1::2]

    # apply 2d rotation to each pair
    # [cos, -sin] [x1] [x1 * cos - x2 * sin]
    # [sin, cos] [x2] = [x1 * sin + x2 * cos]

    rotated = torch.stack([
        x1 * cos - x2 * sin,
        x1 * sin + x2 * cos
    ], dim=-1).flatten(-2)

    return rotated

dim = 8
max_len = 16
cos, sin = precompute_rope_frequencies(dim, max_len)

print("=" * 50)
print("ROPE FREQUENCY VISUALIZATION")
print("=" * 50)
print(f"\nDimension: {dim}, Max length: {max_len}")
print(f"\nRotation angles at position 0: {cos[0].tolist()}")
print(f"Rotation angles at position 1: {cos[1].tolist()}")
print(f"Rotation angles at position 5: {cos[5].tolist()}")


# create simple query and key vector
q = torch.randn(1, 1, 1, dim)  # Query at some position
k = q.clone()  # Same vector as key

def dot_product_with_rope(q, k, pos_q, pos_k, cos, sin):
    q_rot = apply_rope(q, cos[pos_q:pos_q+1], sin[pos_q:pos_q+1])
    k_rot = apply_rope(k, cos[pos_k:pos_k+1], sin[pos_k:pos_k+1])
    return (q_rot * k_rot).sum().item()


# Same distance = same dot product!
print("\nDot products depend only on DISTANCE, not absolute position:")
print(f"  pos(1) · pos(3): {dot_product_with_rope(q, k, 1, 3, cos, sin):.4f}")
print(f"  pos(5) · pos(7): {dot_product_with_rope(q, k, 5, 7, cos, sin):.4f}")
print(f"  pos(10) · pos(12): {dot_product_with_rope(q, k, 10, 12, cos, sin):.4f}")
print("\n  → All ~equal because distance is 2 in each case!")

print(f"\n  pos(1) · pos(1): {dot_product_with_rope(q, k, 1, 1, cos, sin):.4f}")
print(f"  pos(1) · pos(5): {dot_product_with_rope(q, k, 1, 5, cos, sin):.4f}")
print(f"  pos(1) · pos(10): {dot_product_with_rope(q, k, 1, 10, cos, sin):.4f}")
print("\n  → Decreases as distance increases!")

ROPE FREQUENCY VISUALIZATION

Dimension: 8, Max length: 16

Rotation angles at position 0: [1.0, 1.0, 1.0, 1.0]
Rotation angles at position 1: [0.5403023362159729, 0.9950041770935059, 0.9999499917030334, 0.9999995231628418]
Rotation angles at position 5: [0.28366219997406006, 0.8775825500488281, 0.9987502694129944, 0.9999874830245972]

Dot products depend only on DISTANCE, not absolute position:
  pos(1) · pos(3): 1.3162
  pos(5) · pos(7): 1.3162
  pos(10) · pos(12): 1.3162

  → All ~equal because distance is 2 in each case!

  pos(1) · pos(1): 2.6092
  pos(1) · pos(5): 1.0342
  pos(1) · pos(10): 0.4525

  → Decreases as distance increases!


Compare position encoding methods

Visualize and compare sinusoidal, learned, and RoPE encodings side-by-side

In [7]:
# position encoding comparison
# see how different methods encode the same positions

import torch
import numpy as np

# sinusoidal encodings (original transformer method)

def sinusoidal_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()

    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
    )

    pe[:, 0::2] = torch.sin(position * div_term) # even dims
    pe[:,1::2] = torch.cos(position * div_term) # odd dims
    return pe

# learned embeddings (gpt 2 style)

class LearnedPositionEmbedding(torch.nn.Module):
    # simple lookup table, position -> vector
    def __init__(self, max_len, d_model):
        super().__init__()
        self.embedding = torch.nn.Embedding(max_len, d_model)
        # intialize with small random values
        torch.nn.init.normal_(self.embedding.weight, std=0.02)

    def forward(self, positions):
        return self.embedding(positions)

# comparison
d_model =  64
max_len = 20

sinusoidal = sinusoidal_encoding(max_len, d_model)
learned = LearnedPositionEmbedding(max_len, d_model)
learned_pe = learned(torch.arange(max_len))

print("\n1. SINUSOIDAL (Vaswani et al., 2017)")
print("-" * 40)
print(f"   Position 0, first 8 dims: {sinusoidal[0, :8].tolist()}")
print(f"   Position 1, first 8 dims: {sinusoidal[1, :8].tolist()}")
print(f"   Position 10, first 8 dims: {sinusoidal[10, :8].tolist()}")

print("\n2. LEARNED (GPT-2 style)")
print("-" * 40)
print(f"   Position 0, first 8 dims: {learned_pe[0, :8].tolist()}")
print(f"   Position 1, first 8 dims: {learned_pe[1, :8].tolist()}")
print("   (Random initialization - would be trained)")


# Key insight: Sinusoidal has structure, learned is random
print("\n" + "=" * 55)
print("KEY DIFFERENCES")
print("=" * 55)

# Distance preservation in sinusoidal
def cosine_sim(a, b):
    return (a @ b) / (a.norm() * b.norm())

print("\nSINUSOIDAL - Position similarity by distance:")
print(f"   sim(0, 1): {cosine_sim(sinusoidal[0], sinusoidal[1]):.3f}")
print(f"   sim(0, 5): {cosine_sim(sinusoidal[0], sinusoidal[5]):.3f}")
print(f"   sim(0, 10): {cosine_sim(sinusoidal[0], sinusoidal[10]):.3f}")
print(f"   sim(5, 6): {cosine_sim(sinusoidal[5], sinusoidal[6]):.3f}")
print("   → Nearby positions are more similar!")

# Extrapolation test
print("\n" + "=" * 55)
print("EXTRAPOLATION TEST")
print("=" * 55)
print("\nSinusoidal at position 1000 (beyond training):")
extended_sin = sinusoidal_encoding(1001, d_model)
print(f"   Position 1000 exists: {extended_sin[1000, :4].tolist()}")

print("\nLearned at position 1000 (beyond max_len=20):")
print("   CRASH! No embedding exists for position > 19")
print("   This is why learned positions fell out of favor.")

print("\n" + "=" * 55)
print("MODERN CHOICE: RoPE")
print("=" * 55)
print("""
RoPE (used by Llama, Mistral, Qwen, etc.) combines:
✓ Structured like sinusoidal (generalizes)
✓ Effective like learned (high quality)
✓ Encodes RELATIVE position (what attention needs)
✓ Extendable via interpolation/NTK scaling
""")





1. SINUSOIDAL (Vaswani et al., 2017)
----------------------------------------
   Position 0, first 8 dims: [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
   Position 1, first 8 dims: [0.8414709568023682, 0.5403023362159729, 0.6815613508224487, 0.7317609786987305, 0.5331684350967407, 0.8460090756416321, 0.40930891036987305, 0.9123958945274353]
   Position 10, first 8 dims: [-0.5440211296081543, -0.83907151222229, 0.93763267993927, 0.3476276099681854, -0.6129369139671326, 0.7901318669319153, -0.879767119884491, -0.4754049479961395]

2. LEARNED (GPT-2 style)
----------------------------------------
   Position 0, first 8 dims: [-0.020256774500012398, 0.016652893275022507, -0.012989646755158901, -0.014537246897816658, -0.00312043190933764, -0.0179680734872818, 0.011220273561775684, 0.019261684268712997]
   Position 1, first 8 dims: [0.011302906088531017, -0.01524412166327238, 0.01482250913977623, -0.005202078260481358, -0.012239326722919941, -0.012121682986617088, 0.004677274264395237, 0.041580